# Implementation of the example in Appendix C of the thesis of Antoine Dumas:

## Application appui plan et deux goupilles

In [2]:
import math
import sympy as sp
import numpy as np
from typing import List, Tuple
from functools import partial

import otaf

Since the implementation in dumas work is a bit different and that we don't really care about the linerization rn, we have to ceate some intermediary functions for the optimization.

We have multiple sets of variables, and the optimization is only done on the gap varibles (g), meaning we need to have a first function that fixes the other variables (using partial) or something, so that we have in the end only a function that takes as an input the gaps.

In [9]:
# The gap vector is of dimension 16 + 1 (due to the added slack variable)
otaf.example_models.model_dumas_cython.x_full_labels


['w_1a1',
 'alpha_1a1',
 'beta_1a1',
 'w_2a2',
 'alpha_2a2',
 'beta_2a2',
 'u_1b1',
 'v_1b1',
 'alpha_1b1',
 'beta_1b1',
 'u_2b2',
 'v_2b2',
 'alpha_2b2',
 'beta_2b2',
 'u_1c1',
 'v_1c1',
 'alpha_1c1',
 'beta_1c1',
 'u_2c2',
 'v_2c2',
 'alpha_2c2',
 'beta_2c2',
 'd_1b',
 'd_3b',
 'd_1c',
 'd_4c',
 'u_1g1',
 'v_1g1',
 'u_2g2',
 'v_2g2']

In [7]:
def get_compatibility_constraints(x_full, L, d_max):
    return partial(otaf.example_models.model_dumas_cython.constraints_eq_base, x_full=x_full, L=L, d_max=d_max)

def get_interface_constraints(x_full, L, d_max):
    return partial(otaf.example_models.model_dumas_cython.constraints_ineq_slack, x_full=x_full, L=L, d_max=d_max)

def get_optim_functions(x_full, L, d_max):
    return get_compatibility_constraints(x_full, L, d_max), get_interface_constraints(x_full, L, d_max)

### Original values from dumas work (we choose parameter set 1)

In [6]:
L = [100, 40, 30, 30, 20, 20, 120, 50, 40, 50, -30]
d_max = 0.25

mu_d_ext = 20
sigma_d_ext = 0.06

mu_d_int = 19.8
sigma_d_int = 0.06

mu_trans = 0
sigma_trans = 0.01
mu_rot = 0 
sigma_rot = 0.001

In [6]:
deviation_symbols = otaf.example_models.model_dumas_cython.x_full_labels

In [4]:
# Here we have a distribution in the standard space. 
defect_distribution = otaf.distribution.get_composed_normal_defect_distribution(deviation_symbols)
sample_U_50000 = np.array(defect_distribution.getSample(50000))

### Now lets check if we get the same values than A. Dumas with his different parameter values.